# Diagnostik — Checkpoint ve Prediction Sağlık Kontrolü
Checkpoint dosyalarının ve wf_predictions_stage1.pkl'nin durumunu kontrol eder.

In [ ]:
!pip install -q pyyaml pyarrow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, pickle
import numpy as np
import pandas as pd

CHECKPOINT_DIR = '/content/drive/MyDrive/scalp2/checkpoints'
DATA_DIR = '/content/drive/MyDrive/scalp2/data/processed'

# 1. Check all fold directories
print('=' * 70)
print('  CHECKPOINT HEALTH CHECK')
print('=' * 70)

fold_dirs = sorted([d for d in os.listdir(CHECKPOINT_DIR) if d.startswith('fold_')])
print(f'Total fold directories: {len(fold_dirs)}')

feature_counts = {}
broken_folds = []

for fold_name in fold_dirs:
    fold_path = os.path.join(CHECKPOINT_DIR, fold_name)
    fold_idx = int(fold_name.split('_')[1])
    
    # Check feature_names.json
    fn_path = os.path.join(fold_path, 'feature_names.json')
    if os.path.exists(fn_path):
        with open(fn_path, 'r') as f:
            feat_names = json.load(f)
        n_feat = len(feat_names)
    else:
        n_feat = 'MISSING'
        broken_folds.append(fold_idx)
    
    # Check model file
    model_path = os.path.join(fold_path, 'hybrid_encoder.pt')
    model_size = os.path.getsize(model_path) if os.path.exists(model_path) else 0
    
    # Check scaler
    scaler_path = os.path.join(fold_path, 'scaler.pkl')
    scaler_exists = os.path.exists(scaler_path)
    
    # Check XGBoost (from NB5)
    xgb_path = os.path.join(CHECKPOINT_DIR, f'xgb_fold_{fold_idx:03d}.json')
    xgb_exists = os.path.exists(xgb_path)
    
    feature_counts[fold_idx] = n_feat
    
    if fold_idx % 10 == 0 or n_feat != list(feature_counts.values())[0]:
        print(f'  Fold {fold_idx:3d}: features={n_feat}, model={model_size//1024}KB, '
              f'scaler={"OK" if scaler_exists else "MISSING"}, '
              f'xgb={"YES" if xgb_exists else "no"}')

# Summary
unique_counts = set(v for v in feature_counts.values() if v != 'MISSING')
print(f'\nUnique feature counts across folds: {unique_counts}')
if len(unique_counts) > 1:
    print('*** WARNING: Feature count mismatch across folds! ***')
    for count in unique_counts:
        folds_with = [k for k, v in feature_counts.items() if v == count]
        print(f'  {count} features: folds {folds_with[:5]}... ({len(folds_with)} total)')

# Current feature_columns.json
fc_path = os.path.join(DATA_DIR, 'feature_columns.json')
with open(fc_path, 'r') as f:
    current_features = json.load(f)
print(f'\nCurrent feature_columns.json: {len(current_features)} features')
print(f'Checkpoint feature counts match current: {len(current_features) in unique_counts}')

In [ ]:
# 2. Check wf_predictions_stage1.pkl
print('=' * 70)
print('  PREDICTION FILE CHECK')
print('=' * 70)

pkl_path = os.path.join(DATA_DIR, 'wf_predictions_stage1.pkl')
if not os.path.exists(pkl_path):
    print('wf_predictions_stage1.pkl NOT FOUND!')
else:
    with open(pkl_path, 'rb') as f:
        wf_preds = pickle.load(f)
    
    print(f'Total folds in pkl: {len(wf_preds)}')
    
    # Load df for date mapping
    df = pd.read_parquet(os.path.join(DATA_DIR, 'BTC_USDT_labeled.parquet'))
    
    total_bars = 0
    for pred in wf_preds:
        fold_idx = pred['fold_idx']
        ts = pred['test_start']
        te = pred['test_end']
        n_preds = len(pred['test_probabilities'])
        total_bars += n_preds
        
        start_date = df.index[ts] if ts < len(df) else 'OUT_OF_RANGE'
        end_date = df.index[min(te-1, len(df)-1)] if te <= len(df) else 'OUT_OF_RANGE'
        
        has_regime = pred.get('regime_probs') is not None
        
        if fold_idx % 10 == 0 or fold_idx == len(wf_preds) - 1:
            print(f'  Fold {fold_idx:3d}: bars={ts}-{te} ({n_preds} preds), '
                  f'dates={start_date} -> {end_date}, '
                  f'regime={"YES" if has_regime else "NO"}')
    
    print(f'\nTotal prediction bars: {total_bars:,}')
    print(f'Expected (~67 folds * ~2816 bars): ~188,672')
    print(f'Coverage: {total_bars/188672*100:.1f}%')
    
    # Date range
    first_fold = wf_preds[0]
    last_fold = wf_preds[-1]
    first_date = df.index[first_fold['test_start']]
    last_date = df.index[min(last_fold['test_end']-1, len(df)-1)]
    print(f'\nDate coverage: {first_date} -> {last_date}')

In [ ]:
# 3. Quick model compatibility check
import torch
import sys
REPO_DIR = '/content/scalp2_repo'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    !git -C {REPO_DIR} fetch && git -C {REPO_DIR} reset --hard origin/main
else:
    !git clone https://github.com/umutergul74/Scalp2.git {REPO_DIR}
sys.path.insert(0, REPO_DIR)

from scalp2.config import load_config
from scalp2.models.hybrid import HybridEncoder
from scalp2.utils.serialization import load_fold_artifacts

config = load_config(f'{REPO_DIR}/config.yaml')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('=' * 70)
print('  MODEL COMPATIBILITY CHECK')
print('=' * 70)

ok_folds = []
bad_folds = []

for fold_idx in range(67):
    try:
        artifacts = load_fold_artifacts(CHECKPOINT_DIR, fold_idx, device=device)
        n_feat = len(artifacts['feature_names'])
        model = HybridEncoder(n_feat, config.model).to(device)
        model.load_state_dict(artifacts['model_state'])
        ok_folds.append((fold_idx, n_feat))
    except Exception as e:
        bad_folds.append((fold_idx, str(e)[:80]))

print(f'OK folds: {len(ok_folds)} / 67')
print(f'Bad folds: {len(bad_folds)} / 67')

if bad_folds:
    print('\nBroken folds:')
    for idx, err in bad_folds[:10]:
        print(f'  Fold {idx}: {err}')

if ok_folds:
    ok_feats = set(n for _, n in ok_folds)
    print(f'\nFeature dimensions in OK folds: {ok_feats}')

print('\nDone.')